# Lesson 4: Command Runner

Add `run_command` so the agent can execute shell commands — run tests, check git status, install packages, etc.

In [ ]:
%pip install openai-agents --upgrade

<div style='margin-top: 20px; margin-bottom: 20px; text-align: center; display: flex; flex-direction: column; align-items: center;'>
<img src='./images/lesson_4_command_runner.png' width='600' /> 
</div>

In [ ]:
import os
import subprocess
from getpass import getpass
from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [4]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"


@function_tool
def list_files(path: str = ".") -> str:
    """List all files and directories at the given path. Directories end with /."""
    try:
        entries = []
        for entry in sorted(os.listdir(path)):
            full = os.path.join(path, entry)
            if os.path.isdir(full):
                entries.append(f"{entry}/")
            else:
                entries.append(entry)
        return "\n".join(entries) if entries else "(empty directory)"
    except FileNotFoundError:
        return f"Error: Directory not found: {path}"
    except Exception as e:
        return f"Error listing files: {e}"

## Defining the `run_command` Tool

This is the most powerful tool — it lets the agent run arbitrary shell commands. We add a timeout and output limit for safety.

In [5]:
@function_tool
def run_command(command: str) -> str:
    """Run a shell command and return its output. Use this for running tests, checking git status, or any system command."""
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )
        output = result.stdout
        if result.stderr:
            output += f"\nSTDERR:\n{result.stderr}"
        if result.returncode != 0:
            output += f"\n(exit code: {result.returncode})"
        if len(output) > 10000:
            output = output[:10000] + "\n... (truncated)"
        return output if output.strip() else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds"
    except Exception as e:
        return f"Error running command: {e}"

In [6]:
agent = Agent(
    name="Command Runner",
    instructions="You are a coding assistant that can explore files and run shell commands.",
    model=MODEL,
    tools=[read_file, list_files, run_command],
)

In [7]:
result = await Runner.run(agent, "What version of Python is installed? And what's the current git branch?")
print(result.final_output)

Python 3.12.4 is installed, and the current git branch is `main`.


## Summary

- Added `run_command` with timeout and output limits for safety.
- The agent can now explore files *and* interact with the system.
- Next lesson: add `edit_file` so the agent can modify code.